This notebook is to fine-tune a pre-trained model and extract metadata from survey plans

In [35]:

import os
import sys
from pathlib import Path
import pandas as pd
from box import Box
import yaml

from huggingface_hub import snapshot_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [36]:
def load_config(file_path):
    """Load a YAML configuration file into a Box for attribute access."""
    with open(file_path, "r") as file:
        return Box(yaml.safe_load(file))

In [33]:
"""
Download base VLM checkpoints from HuggingFace for Unsloth fine-tuning.

Downloads Qwen2.5-VL-3B to models/ directory.
Uses HF transfer for faster downloads.
"""

BASE_CONFIG = load_config("/workspace/datadata/base.yaml")

ROOT_DIR = Path(BASE_CONFIG["root_dir"])
MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

repo_ids = [
    "Qwen/Qwen2.5-VL-3B-Instruct"
]

for repo_id in repo_ids:
    local_dir = MODELS_DIR / repo_id.split("/")[-1]
    local_dir.mkdir(parents=True, exist_ok=True)
    path = snapshot_download(
        repo_id=repo_id,
        repo_type="model",
        local_dir=local_dir,
        local_dir_use_symlinks=False,
        resume_download=True,  # resumable
        max_workers=16,
    )
    print(f"downloaded: {repo_id} -> {path}")

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

downloaded: Qwen/Qwen2.5-VL-3B-Instruct -> /workspace/models/Qwen2.5-VL-3B-Instruct


In [37]:
BASE_CONFIG = load_config("/workspace/datadata/base.yaml")

ROOT_DIR = Path(BASE_CONFIG["root_dir"])
DATA_DIR = ROOT_DIR / "datadata"
IMAGES_DIR = DATA_DIR / "survey_plans"

SEED = BASE_CONFIG["seed"]    # Random seed for reproducibility

train = pd.read_csv(DATA_DIR / "Train.csv")
test = pd.read_csv(DATA_DIR / "Test.csv")


train = train.drop_duplicates(subset=["ID"]).reset_index(drop=True)

# Map image IDs to file paths
imgs_df = pd.Series(list(IMAGES_DIR.glob("*.jpg"))).to_frame(name="image_path")
imgs_df["ID"] = imgs_df["image_path"].apply(lambda x: x.stem.split("_")[-1])

# Merge all data sources
df = pd.concat(
    [train.assign(split="train"), test.assign(split="test")], ignore_index=True
)
df = df.merge(imgs_df, on="ID", how="left")        # Add image paths

In [38]:
df.to_csv(DATA_DIR / "df.csv", index=False)

In [39]:
import argparse
import ast
import json
import os
import re
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import yaml

In [40]:
os.environ["BNB_CUDA_VERSION"] = ""
os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"

In [ ]:
# Configuration
def load_config(path: str) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)

In [ ]:
# Prompts
LABEL_CORRECTION_PROMPT = """You are given 7 image tiles of a Barbados survey plan document and a JSON object containing noisy/OCR-extracted metadata labels.

Your task: Verify each field against what you can read in the images. If a field is incorrect or garbled, correct it. If correct, keep it unchanged. If a field cannot be determined from the images, set it to null.

Return ONLY a valid JSON object with these exact keys:
- Land Surveyor
- Surveyed For
- Certified date
- Total Area
- Unit of Measurement
- Address
- LT Num

No additional text, no markdown, no explanation. Just the JSON object.

Noisy labels:
"""

TRAINING_PROMPT = """You are given 7 image tiles of a Barbados survey plan document:
1. Full document
2. Bottom half
3. Bottom-left quadrant
4. Bottom-right quadrant
5. Top half
6. Top-left quadrant
7. Top-right quadrant

Extract the following fields from the survey plan. If a field is not visible or cannot be determined, use "N/A".

Fields to extract:
- Land Surveyor: Primary surveyor name
- Land Surveyor2: Secondary surveyor name if present
- Surveyed For: Name of person/entity the survey was conducted for
- Certified date: Date the survey was certified
- Total Area: Numeric area value
- Unit of Measurement: Unit for the area (e.g., sq ft, acres, perches)
- Address: Property address
- Address2: Secondary address if present
- Parish: Parish name
- LT Num: LT registration number

Return the extracted information in the format:
Land Surveyor: [value]
Land Surveyor2: [value]
Surveyed For: [value]
Certified date: [value]
Total Area: [value]
Unit of Measurement: [value]
Address: [value]
Address2: [value]
Parish: [value]
LT Num: [value]
"""

JSON_INSTRUCTION = (
    "Respond with ONLY a single valid JSON object. "
    "No prose, no markdown, no code fences. "
    "Quote all string values (e.g., '7A' must be \"7A\")."
)

In [ ]:
# Stage 1: Patchify Images
def patchify_images(cfg: dict) -> None:
    """Create 7-tile patches from survey plan images."""
    import pandas as pd
    from PIL import Image, ImageFile
    from tqdm.auto import tqdm

    ImageFile.LOAD_TRUNCATED_IMAGES = True
    Image.MAX_IMAGE_PIXELS = None

    data_dir = Path(cfg["paths"]["data_dir"])
    df = pd.read_csv(data_dir / cfg["files"]["input_csv"])
    df = df.drop_duplicates(subset=["ID"], ignore_index=True)
    df = df[~df.image_path.isna()].reset_index(drop=True)

    def letterbox_square(img: Image.Image, target: int, fill=(255, 255, 255)) -> Image.Image:
        w, h = img.size
        s = target / max(w, h)
        nw, nh = max(1, int(round(w * s))), max(1, int(round(h * s)))
        im = img.resize((nw, nh), Image.LANCZOS)
        canvas = Image.new("RGB", (target, target), fill)
        canvas.paste(im, ((target - nw) // 2, (target - nh) // 2))
        return canvas

    side_to_col = {
        "full": "full_path",
        "top": "top_path",
        "bottom": "bottom_path",
        "top_left": "tl_path",
        "top_right": "tr_path",
        "bottom_left": "bl_path",
        "bottom_right": "br_path",
    }

    def process_one(rec, out_dir: Path, size: int):
        out_rows = []
        ID = str(rec["ID"])
        img_path = Path(str(rec["image_path"]))
        try:
            with Image.open(img_path) as im0:
                img = im0.convert("RGB")
        except Exception:
            return out_rows

        w, h = img.size
        mw, mh = w // 2, h // 2
        tiles = {
            "full": img,
            "top": img.crop((0, 0, w, mh)),
            "bottom": img.crop((0, mh, w, h)),
            "top_left": img.crop((0, 0, mw, mh)),
            "top_right": img.crop((mw, 0, w, mh)),
            "bottom_left": img.crop((0, mh, mw, h)),
            "bottom_right": img.crop((mw, mh, w, h)),
        }

        id_dir = out_dir / ID
        id_dir.mkdir(parents=True, exist_ok=True)

        for side, tile in tiles.items():
            save_path = id_dir / f"{side}_{size}x{size}.png"
            try:
                letterbox_square(tile, size).save(save_path)
                out_rows.append({"ID": ID, "col": side_to_col[side], "image_path": str(save_path)})
            except Exception:
                continue
        return out_rows

    for size in cfg["images"]["sizes"]:
        out_dir = data_dir / f"patched_{size}"
        out_dir.mkdir(parents=True, exist_ok=True)
        csv_path = data_dir / cfg["files"]["patched_csv"].format(size=size)

        records = df[["ID", "image_path"]].to_dict("records")
        rows = []
        max_workers = min(cfg["processing"]["max_workers"], os.cpu_count() or 4)

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = [ex.submit(process_one, r, out_dir, size) for r in records]
            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Patching {size}px"):
                rows.extend(fut.result())

        if rows:
            paths_df = pd.DataFrame(rows)
            wide = paths_df.pivot_table(index="ID", columns="col", values="image_path", aggfunc="first").reset_index()
            wide.columns.name = None
            base = df.copy()
            if "geometry" in base.columns:
                base = base.drop(columns=["geometry"])
            patched_df = base.merge(wide, on="ID", how="left")
            patched_df.to_csv(csv_path, index=False)
            print(f"Saved: {csv_path}")

In [ ]:
# Stage 2: Automated Label Correction
def correct_labels(cfg: dict) -> None:
    """Use base VLM to correct noisy training labels."""
    import pandas as pd
    import torch
    from PIL import Image
    from tqdm.auto import tqdm
    from unsloth import FastVisionModel

    os.environ["CUDA_VISIBLE_DEVICES"] = cfg["processing"]["cuda_device"]

    data_dir = Path(cfg["paths"]["data_dir"])
    models_dir = Path(cfg["paths"]["models_dir"])
    target_size = cfg["images"]["target_size"]

    df = pd.read_csv(data_dir / cfg["files"]["patched_csv"].format(size=target_size))
    train_df = df[df["split"] == "train"].drop_duplicates(subset=["ID"]).reset_index(drop=True)

    checkpoint = models_dir / cfg["model"]["repo_id"].split("/")[-1]
    model, tokenizer = FastVisionModel.from_pretrained(
        str(checkpoint),
        load_in_4bit=cfg["model"]["load_in_4bit"],
        use_gradient_checkpointing=cfg["model"]["use_gradient_checkpointing"],
    )

    lora_cfg = cfg["lora"]
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=lora_cfg["finetune_vision_layers"],
        finetune_language_layers=lora_cfg["finetune_language_layers"],
        finetune_attention_modules=lora_cfg["finetune_attention_modules"],
        finetune_mlp_modules=lora_cfg["finetune_mlp_modules"],
        r=lora_cfg["r"],
        lora_alpha=lora_cfg["alpha"],
        lora_dropout=0,
        bias=lora_cfg["bias"],
        random_state=lora_cfg["random_state"],
        use_rslora=lora_cfg["use_rslora"],
        loftq_config=None,
    )

    FastVisionModel.for_inference(model)
    device = next(model.parameters()).device
    float_dtypes = [p.dtype for p in model.parameters() if p.is_floating_point()]
    model_dtype = float_dtypes[0] if float_dtypes else torch.float16
    use_amp = device.type == "cuda" and model_dtype in (torch.float16, torch.bfloat16)

    def open_rgb(path):
        with Image.open(path) as img:
            return img.convert("RGB")

    def row_to_labels_json(row):
        obj = {
            "Land Surveyor": str(row["Land Surveyor"]),
            "Surveyed For": str(row["Surveyed For"]),
            "Certified date": str(row["Certified date"]),
            "Total Area": float(row["Total Area"]) if pd.notna(row["Total Area"]) else 0.0,
            "Unit of Measurement": str(row["Unit of Measurement"]),
            "Address": str(row["Address"]),
            "LT Num": str(row["LT Num"]),
        }
        return json.dumps(obj, ensure_ascii=False)

    def build_request(row):
        images = [
            {"type": "image", "image": open_rgb(row["full_path"])},
            {"type": "image", "image": open_rgb(row["bottom_path"])},
            {"type": "image", "image": open_rgb(row["bl_path"])},
            {"type": "image", "image": open_rgb(row["br_path"])},
            {"type": "image", "image": open_rgb(row["top_path"])},
            {"type": "image", "image": open_rgb(row["tl_path"])},
            {"type": "image", "image": open_rgb(row["tr_path"])},
        ]
        return {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": LABEL_CORRECTION_PROMPT},
                        {"type": "text", "text": row_to_labels_json(row)},
                        *images,
                    ],
                }
            ]
        }

    def extract_json(txt: str):
        m = re.search(r"\{.*\}", txt, flags=re.S)
        if not m:
            return {}
        try:
            return json.loads(m.group(0))
        except Exception:
            return {}

    def to_device_dtype(batch, device, dtype):
        out = {}
        for k, v in batch.items():
            if torch.is_tensor(v):
                v = v.to(device)
                if torch.is_floating_point(v):
                    v = v.to(dtype)
            out[k] = v
        return out

    requests = [build_request(r) for _, r in train_df.iterrows()]
    fields = cfg["fields"]["correction"]
    df_out = train_df.copy()
    for f in fields:
        df_out[f + "_corr"] = None

    batch_size = 1
    max_tokens = cfg["training"]["label_correction"]["max_new_tokens"]
    temperature = cfg["training"]["label_correction"]["temperature"]
    csv_path = data_dir / cfg["files"]["label_corrections_csv"]

    if csv_path.exists():
        csv_path.unlink()

    for start in tqdm(range(0, len(requests), batch_size), desc="Correcting labels"):
        batch = requests[start : start + batch_size]
        texts, images_batch = [], []

        for req in batch:
            msgs = req["messages"]
            texts.append(tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False))
            content = msgs[0]["content"]
            images_batch.append([c["image"] for c in content if isinstance(c, dict) and c.get("type") == "image"])

        inputs = tokenizer(text=texts, images=images_batch, add_special_tokens=False, padding=True, return_tensors="pt")
        inputs = to_device_dtype(inputs, device, model_dtype)

        with torch.no_grad():
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=model_dtype):
                    out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, temperature=temperature, use_cache=True)
            else:
                out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, temperature=temperature, use_cache=True)

        input_lens = inputs["attention_mask"].sum(dim=1).tolist() if "attention_mask" in inputs else [inputs["input_ids"].shape[1]] * len(batch)

        for i in range(len(batch)):
            gen_ids = out[i][int(input_lens[i]) :]
            gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            pred = extract_json(gen_text)
            df_idx = df_out.index[start + i]
            for f in fields:
                df_out.at[df_idx, f + "_corr"] = pred.get(f, None)

        written = df_out.iloc[start : start + len(batch)].copy()
        written["Total Area_corr"] = pd.to_numeric(written["Total Area_corr"], errors="coerce")
        written.to_csv(csv_path, mode="a", header=not csv_path.exists() or csv_path.stat().st_size == 0, index=False)

    print(f"Saved: {csv_path}")

In [ ]:
# Stage 3: Training
def train_model(cfg: dict, stage: str) -> None:
    """Fine-tune Qwen-VL on labels."""
    import pandas as pd
    import torch
    from PIL import Image
    from tqdm.auto import tqdm
    from unsloth import FastVisionModel
    from unsloth.trainer import UnslothVisionDataCollator
    from trl import SFTConfig, SFTTrainer

    os.environ["CUDA_VISIBLE_DEVICES"] = cfg["processing"]["cuda_device"]

    data_dir = Path(cfg["paths"]["data_dir"])
    models_dir = Path(cfg["paths"]["models_dir"])
    output_dir = Path(cfg["paths"]["output_dir"])
    target_size = cfg["images"]["target_size"]

    patched_df = pd.read_csv(data_dir / cfg["files"]["patched_csv"].format(size=target_size))
    corr_df = pd.read_csv(data_dir / cfg["files"]["label_corrections_csv"])

    patched_df = patched_df.merge(
        corr_df[["ID", "Land Surveyor_corr", "Surveyed For_corr", "Certified date_corr",
                    "Total Area_corr", "Unit of Measurement_corr", "Address_corr", "LT Num_corr"]],
        on="ID", how="left"
    )
    train_df = patched_df[patched_df.split == "train"].reset_index(drop=True)

    checkpoint = models_dir / cfg["model"]["repo_id"].split("/")[-1]
    model, tokenizer = FastVisionModel.from_pretrained(
        str(checkpoint),
        load_in_4bit=cfg["model"]["load_in_4bit"],
        use_gradient_checkpointing=cfg["model"]["use_gradient_checkpointing"],
    )

    lora_cfg = cfg["lora"]
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=lora_cfg["finetune_vision_layers"],
        finetune_language_layers=lora_cfg["finetune_language_layers"],
        finetune_attention_modules=lora_cfg["finetune_attention_modules"],
        finetune_mlp_modules=lora_cfg["finetune_mlp_modules"],
        r=lora_cfg["r"],
        lora_alpha=lora_cfg["alpha"],
        lora_dropout=lora_cfg["dropout"],
        bias=lora_cfg["bias"],
        random_state=lora_cfg["random_state"],
        use_rslora=lora_cfg["use_rslora"],
        loftq_config=None,
    )

    def open_rgb(path):
        with Image.open(path) as img:
            return img.convert("RGB")

    def row_to_answer(row):
        return (
            f"Land Surveyor: {row['Land Surveyor_corr']}\n"
            f"Land Surveyor2: {row['Land Surveyor']}\n"
            f"Surveyed For: {row['Surveyed For_corr']}\n"
            f"Certified date: {row['Certified date_corr']}\n"
            f"Total Area: {row['Total Area_corr']}\n"
            f"Unit of Measurement: {row['Unit of Measurement_corr']}\n"
            f"Address: {row['Address_corr']}\n"
            f"Address2: {row['Address']}\n"
            f"Parish: {row['Parish']}\n"
            f"LT Num: {row['LT Num_corr']}"
        )

    def row_to_conversation(row):
        images = [
            {"type": "image", "image": open_rgb(row["full_path"])},
            {"type": "image", "image": open_rgb(row["bottom_path"])},
            {"type": "image", "image": open_rgb(row["bl_path"])},
            {"type": "image", "image": open_rgb(row["br_path"])},
            {"type": "image", "image": open_rgb(row["top_path"])},
            {"type": "image", "image": open_rgb(row["tl_path"])},
            {"type": "image", "image": open_rgb(row["tr_path"])},
        ]
        return {
            "messages": [
                {"role": "user", "content": [{"type": "text", "text": TRAINING_PROMPT}, *images]},
                {"role": "assistant", "content": [{"type": "text", "text": row_to_answer(row)}]},
            ]
        }

    train_dataset = [row_to_conversation(r) for _, r in train_df.iterrows()]
    FastVisionModel.for_training(model)

    max_length = cfg["inference"]["max_seq_length"]
    tokenizer.truncation_side = "left"
    tokenizer.model_max_length = max_length

    train_cfg = cfg["training"][stage]
    trainer = SFTTrainer(
        model=model,
        data_collator=UnslothVisionDataCollator(model, tokenizer, max_seq_length=max_length, resize=cfg["inference"]["collator_resize"]),
        train_dataset=train_dataset,
        args=SFTConfig(
            per_device_train_batch_size=train_cfg["per_device_batch_size"],
            gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
            bf16=train_cfg["bf16"],
            tf32=train_cfg["tf32"],
            gradient_checkpointing=True,
            max_grad_norm=train_cfg["max_grad_norm"],
            warmup_ratio=train_cfg["warmup_ratio"],
            learning_rate=train_cfg["learning_rate"],
            logging_steps=1,
            optim="paged_adamw_8bit",
            weight_decay=train_cfg["weight_decay"],
            lr_scheduler_type=train_cfg["lr_scheduler"],
            max_steps=train_cfg["max_steps"],
            seed=cfg["processing"]["seed"],
            output_dir=str(output_dir),
            report_to="none",
            remove_unused_columns=False,
            dataset_text_field=None,
            dataset_kwargs={"skip_prepare_dataset": True},
            neftune_noise_alpha=train_cfg["neftune_noise_alpha"],
            packing=False,
        ),
    )

    trainer.train()

    adapter_dir = output_dir / cfg["files"]["final_model_dir"]
    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    print(f"Model saved to: {adapter_dir}")

    return model, tokenizer

In [ ]:
# Stage 4: Inference
def inference(cfg: dict, model=None, tokenizer=None) -> None:
    """Inference."""
    import pandas as pd
    import torch
    from PIL import Image
    from tqdm.auto import tqdm
    from unsloth import FastVisionModel
    from peft import PeftModel

    os.environ["CUDA_VISIBLE_DEVICES"] = cfg["processing"]["cuda_device"]

    data_dir = Path(cfg["paths"]["data_dir"])
    models_dir = Path(cfg["paths"]["models_dir"])
    target_size = cfg["images"]["target_size"]
    adapter_dir = Path(cfg["paths"]["output_dir"]) / cfg["files"]["final_model_dir"]

    patched_df = pd.read_csv(data_dir / cfg["files"]["patched_csv"].format(size=target_size))
    test_df = patched_df[patched_df.split == "test"].reset_index(drop=True)

    if model is None or tokenizer is None:
        checkpoint = models_dir / cfg["model"]["repo_id"].split("/")[-1]
        model, tokenizer = FastVisionModel.from_pretrained(
            str(checkpoint),
            load_in_4bit=cfg["model"]["load_in_4bit"],
            use_gradient_checkpointing=cfg["model"]["use_gradient_checkpointing"],
        )
        
        # Load trained LoRA adapter
        model = PeftModel.from_pretrained(model, str(adapter_dir))

    FastVisionModel.for_inference(model)
    model.eval()

    device = next(model.parameters()).device
    float_dtypes = [p.dtype for p in model.parameters() if p.is_floating_point()]
    model_dtype = float_dtypes[0] if float_dtypes else torch.float16

    def open_rgb(path):
        with Image.open(path) as img:
            return img.convert("RGB")

    def extract_first_json(s: str):
        s = s.strip()
        s = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*\n", "", s)
        s = re.sub(r"\n\s*```\s*$", "", s)
        start, depth, in_str, esc, quote = None, 0, False, False, None
        for i, ch in enumerate(s):
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == quote:
                    in_str = False
            else:
                if ch in ("'", '"'):
                    in_str, quote = True, ch
                elif ch == "{":
                    if depth == 0:
                        start = i
                    depth += 1
                elif ch == "}":
                    if depth > 0:
                        depth -= 1
                        if depth == 0 and start is not None:
                            return s[start : i + 1]
        if "{" in s and "}" in s:
            i, j = s.find("{"), s.rfind("}")
            if i < j:
                return s[i : j + 1]
        return None

    def repair_jsonish(s: str):
        s = s.replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'")
        s = re.sub(r",\s*(?=[}\]])", "", s)
        s = re.sub(r"\bTrue\b", "true", s)
        s = re.sub(r"\bFalse\b", "false", s)
        s = re.sub(r"\bNone\b", "null", s)
        s = re.sub(r"([{,]\s*)([A-Za-z_][A-Za-z0-9_\- ]*)(\s*):", r'\1"\2"\3:', s)
        return s

    def parse_json(text: str):
        cand = extract_first_json(text) or text.strip()
        try:
            return json.loads(cand)
        except Exception:
            pass
        try:
            val = ast.literal_eval(cand)
            if isinstance(val, dict):
                return val
        except Exception:
            pass
        repaired = repair_jsonish(cand)
        try:
            return json.loads(repaired)
        except Exception:
            pass
        data = {}
        for line in text.splitlines():
            line = line.strip()
            if not line or line.startswith(("#", "//", "*")):
                continue
            if ":" in line:
                k, v = line.split(":", 1)
                k = k.strip().strip('"').strip("'")
                v = v.strip().rstrip(",")
                if k:
                    data[k] = v
        return data

    pred_rows, all_keys = [], []
    max_tokens = cfg["inference"]["max_new_tokens"]

    for row in tqdm(test_df.itertuples(index=True), total=len(test_df), desc="Inference"):
        try:
            images = [
                open_rgb(getattr(row, "full_path")),
                open_rgb(getattr(row, "bottom_path")),
                open_rgb(getattr(row, "bl_path")),
                open_rgb(getattr(row, "br_path")),
                open_rgb(getattr(row, "top_path")),
                open_rgb(getattr(row, "tl_path")),
                open_rgb(getattr(row, "tr_path")),
            ]

            messages = [
                {
                    "role": "user",
                    "content": [{"type": "text", "text": TRAINING_PROMPT + "\n\n" + JSON_INSTRUCTION}]
                    + [{"type": "image"} for _ in range(7)],
                }
            ]

            input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
            inputs = tokenizer(images, input_text, add_special_tokens=False, return_tensors="pt")

            for k, v in list(inputs.items()):
                if torch.is_tensor(v):
                    v = v.to(device)
                    if torch.is_floating_point(v):
                        v = v.to(model_dtype)
                    inputs[k] = v

            with torch.inference_mode():
                if device.type == "cuda":
                    with torch.autocast(device_type="cuda", dtype=model_dtype):
                        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, use_cache=True)
                else:
                    out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, use_cache=True)

            gen_only = out[0, inputs["input_ids"].shape[1] :]
            text = tokenizer.decode(gen_only, skip_special_tokens=True).strip()
            pred = parse_json(text)

            if not isinstance(pred, dict):
                pred = {}

            pred_rows.append(pred)
            for k in pred.keys():
                if k not in all_keys:
                    all_keys.append(k)

        except Exception:
            pred_rows.append({})

    pred_df = pd.DataFrame.from_records(pred_rows, index=test_df.index)
    pred_df = pred_df.reindex(columns=all_keys)
    pred_df["ID"] = test_df["ID"].values
    path_cols = [col for col in patched_df.columns if col.endswith("_path")]
    pred_df = pred_df.merge(patched_df[["ID"] + path_cols], how="left", on="ID")

    csv_path = data_dir / cfg["files"]["inference_csv"]
    pred_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")

In [51]:
cfg = load_config("/workspace/datadata/config.yaml")

Path(cfg["paths"]["data_dir"]).mkdir(parents=True, exist_ok=True)
Path(cfg["paths"]["output_dir"]).mkdir(parents=True, exist_ok=True)

In [52]:
with open("/workspace/datadata/config.yaml", "r") as f:
    print(f.read())


paths:
  root_dir: /workspace
  data_dir: /workspace/datadata
  models_dir: /workspace/models
  output_dir: /workspace/outputs
files:
  input_csv: df.csv
  patched_csv: patched_{size}.csv
  label_corrections_csv: label_corrections.csv
  inference_csv: test_predictions.csv
  final_model_dir: qwen3vl8b_finetuned_lora
model:
  repo_id: Qwen/Qwen2.5-VL-3B-Instruct
  load_in_4bit: true
  use_gradient_checkpointing: unsloth
lora:
  r: 16
  alpha: 16
  dropout: 0.35
  bias: none
  random_state: 3407
  use_rslora: false
  finetune_vision_layers: true
  finetune_language_layers: true
  finetune_attention_modules: true
  finetune_mlp_modules: true
images:
  sizes:
  - 1024
  - 896
  target_size: 1024
  letterbox_fill:
  - 255
  - 255
  - 255
  tiles:
  - full
  - top
  - bottom
  - top_left
  - top_right
  - bottom_left
  - bottom_right
training:
  label_correction:
    batch_size: 2
    max_new_tokens: 384
    temperature: 0.001
  stage1:
    per_device_batch_size: 2
    gradient_accumulation_s

In [53]:
print("\n[Stage 1] Patchifying images...")
patchify_images(cfg)


[Stage 1] Patchifying images...


Patching 1024px:   0%|          | 0/877 [00:00<?, ?it/s]

Saved: /workspace/datadata/patched_1024.csv


Patching 896px:   0%|          | 0/877 [00:00<?, ?it/s]

Saved: /workspace/datadata/patched_896.csv


In [54]:
import gc
import torch

torch.cuda.empty_cache()
torch.cuda.ipc_collect()
gc.collect() 

80741

In [55]:
print("\n[Stage 2] Correcting labels...")
correct_labels(cfg)


[Stage 2] Correcting labels...
==((====))==  Unsloth 2025.10.3: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Correcting labels:   0%|          | 0/658 [00:00<?, ?it/s]

Saved: /workspace/datadata/label_corrections.csv


In [62]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
gc.collect() 

1736

In [63]:
model, tokenizer = None, None
print("\n[Stage 3] Training on corrected labels...")
model, tokenizer = train_model(cfg, stage="stage1")


[Stage 3] Training on corrected labels...
==((====))==  Unsloth 2025.10.3: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 658 | Num Epochs = 1 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,084,928 of 3,795,707,904 (1.08% trained)


Step,Training Loss
1,0.487800
2,0.483900
3,0.477100
4,0.475000
5,0.463000
6,0.433100
7,0.404800
8,0.386400
9,0.367300
10,0.351500


Model saved to: /workspace/outputs/qwen3vl8b_finetuned_lora


In [65]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
gc.collect() 

431

In [66]:
print("\n[Stage 4] Running test inference...")
inference(cfg, model=model, tokenizer=tokenizer)


[Stage 4] Running test inference...


Inference:   0%|          | 0/219 [00:00<?, ?it/s]

Saved: /workspace/datadata/test_predictions.csv
